In [ ]:
import pandas as pd
import numpy as np
import xlsxwriter
import re
from pathlib import Path

def extraer_complejidad(texto):
    """
    Asumiendo que 'texto' tiene el formato "loops: X, conditionals: Y",
    extrae X y Y y devuelve su suma.
    Si no se encuentra el formato, devuelve np.nan.
    """
    try:
        # Buscar números usando expresiones regulares
        numeros = re.findall(r'\d+', texto)
        if len(numeros) >= 2:
            return float(numeros[0]) + float(numeros[1])
        else:
            return np.nan
    except:
        return np.nan

def main():
    # =========================================================================
    # 1) LECTURA DE DATOS CRUDOS DESDE CSV
    # =========================================================================
    csv_path = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results\results_completo.csv"  # Ajusta la ruta
    df = pd.read_csv(csv_path)
    
    # Se espera que el CSV tenga al menos las siguientes columnas:
    # ID, model, temperature, top_p, top_k, code, result, true_count, false_count, execution_time, memory_usage_MB, algorithmic_complexity

    # =========================================================================
    # 2) CALCULAR COLUMNAS DERIVADAS EN PYTHON (RESULTADOS FIJOS)
    # =========================================================================
    # Total Tests = true_count + false_count
    df["Total Tests"] = df["true_count"] + df["false_count"]
    # Accuracy = true_count / Total Tests (si Total Tests > 0, sino NaN)
    df["Accuracy"] = np.where(df["Total Tests"] > 0, df["true_count"] / df["Total Tests"], np.nan)
    # Numeric Complexity: extraer la suma de los números de la columna algorithmic_complexity
    df["Numeric Complexity"] = df["algorithmic_complexity"].apply(lambda x: extraer_complejidad(str(x)))

    # =========================================================================
    # 3) GENERAR LA HOJA "Raw Data" CON RESULTADOS (valores fijos)
    # =========================================================================
    excel_filename = "Analisis_C.xlsx"
    workbook = xlsxwriter.Workbook(excel_filename)
    raw_sheet = workbook.add_worksheet("Raw Data")
    
    # Columnas originales + nuevas columnas
    columnas = list(df.columns)
    # Escribimos encabezados
    for col_idx, col_name in enumerate(columnas):
        raw_sheet.write(0, col_idx, col_name)
    # Escribimos los datos (valores fijos)
    for r in range(len(df)):
        for c in range(len(df.columns)):
            raw_sheet.write(r+1, c, df.iat[r, c])
    # Ajustar ancho de columnas (opcional)
    raw_sheet.set_column(0, 0, 6)    # ID
    raw_sheet.set_column(1, 1, 18)   # model
    raw_sheet.set_column(2, 4, 10)   # temperature, top_p, top_k
    raw_sheet.set_column(5, 5, 60)   # code
    raw_sheet.set_column(6, 6, 25)   # result
    raw_sheet.set_column(7, 8, 12)   # true_count, false_count
    raw_sheet.set_column(9, 10, 14)  # execution_time, memory_usage_MB
    raw_sheet.set_column(11, 11, 25) # algorithmic_complexity
    raw_sheet.set_column(12, 12, 15) # Total Tests
    raw_sheet.set_column(13, 13, 15) # Accuracy
    raw_sheet.set_column(14, 14, 20) # Numeric Complexity

    # =========================================================================
    # 4) HOJA "Comparisons": AGRUPAR CONFIGURACIONES Y CALCULAR MÉTRICAS
    # =========================================================================
    # Agrupar por model, temperature, top_p y top_k
    agrupado = df.groupby(["model", "temperature", "top_p", "top_k"]).agg({
        "Accuracy": "mean",
        "execution_time": "mean",
        "memory_usage_MB": "mean",
        "Numeric Complexity": "mean"
    }).reset_index()
    # Renombramos columnas para claridad
    agrupado.rename(columns={
        "Accuracy": "Avg Accuracy",
        "execution_time": "Avg Time (s)",
        "memory_usage_MB": "Avg Memory (MB)",
        "Numeric Complexity": "Avg Complexity"
    }, inplace=True)
    # Config Label
    agrupado["Config Label"] = agrupado.apply(lambda row: f"{row['model']} (T:{row['temperature']}, p:{row['top_p']}, k:{row['top_k']})", axis=1)
    
    # Ahora calculamos rankings para cada métrica.
    # Para Accuracy, mayor es mejor: Rank Accuracy (ranking descendente)
    agrupado["Rank Accuracy"] = agrupado["Avg Accuracy"].rank(ascending=False, method="min")
    # Para Time y Memory, menor es mejor:
    agrupado["Rank Time"] = agrupado["Avg Time (s)"].rank(ascending=True, method="min")
    agrupado["Rank Memory"] = agrupado["Avg Memory (MB)"].rank(ascending=True, method="min")
    # Para Complexity, menor es mejor:
    agrupado["Rank Complexity"] = agrupado["Avg Complexity"].rank(ascending=True, method="min")
    # Composite Rank: promedio de los 4 rankings
    agrupado["Composite Rank"] = agrupado[["Rank Accuracy", "Rank Time", "Rank Memory", "Rank Complexity"]].mean(axis=1)
    # Global Best Composite: "Yes" si es el mínimo composite rank global
    best_global = agrupado["Composite Rank"].min()
    agrupado["Global Best Composite?"] = np.where(agrupado["Composite Rank"] == best_global, "Yes", "")
    # Best per Model Composite: "Yes" si, dentro del mismo modelo, es el mínimo
    agrupado["Best per Model Composite?"] = agrupado.groupby("model")["Composite Rank"].transform(lambda x: np.where(x==x.min(),"Yes",""))
    
    # Escribir la hoja "Comparisons" con estos valores
    comp_sheet = workbook.add_worksheet("Comparisons")
    comp_columns = list(agrupado.columns)
    for col_idx, col_name in enumerate(comp_columns):
        comp_sheet.write(0, col_idx, col_name)
    for r in range(len(agrupado)):
        for c in range(len(comp_columns)):
            comp_sheet.write(r+1, c, agrupado.iat[r, c])
    # Ajustar anchos
    comp_sheet.set_column(0, 0, 20)
    comp_sheet.set_column(1, 3, 14)
    comp_sheet.set_column(4, 7, 18)
    comp_sheet.set_column(8, 8, 50)
    comp_sheet.set_column(9, 15, 18)

    # =========================================================================
    # 5) HOJA "Explanation": TEXTO CON DETALLE DE VARIABLES Y MÉTODOS
    # =========================================================================
    expl_sheet = workbook.add_worksheet("Explanation")
    expl_text = (
        "EXPLICACIÓN DETALLADA DE VARIABLES Y CÁLCULOS:\n\n"
        "Hoja 'Raw Data':\n"
        " - Se importan los datos crudos del CSV.\n"
        " - 'Total Tests' = true_count + false_count.\n"
        " - 'Accuracy' = true_count / (true_count + false_count) si Total Tests > 0.\n"
        " - 'Numeric Complexity': Se extraen los números de la cadena de 'algorithmic_complexity',\n"
        "    por ejemplo, si la celda contiene \"loops: 3, conditionals: 2\", se calcula 3+2 = 5.\n\n"
        "Hoja 'Comparisons':\n"
        " - Se agrupa por [model, temperature, top_p, top_k] para calcular los promedios:\n"
        "    • Avg Accuracy, Avg Time (s), Avg Memory (MB) y Avg Complexity.\n"
        " - 'Config Label': concatenación de model y parámetros para identificar la configuración completa.\n"
        " - Se calculan rankings para cada métrica:\n"
        "    • Rank Accuracy: mayor es mejor.\n"
        "    • Rank Time, Rank Memory y Rank Complexity: menor es mejor.\n"
        " - 'Composite Rank': promedio de los 4 rankings, para combinar las métricas en una puntuación global.\n"
        " - 'Global Best Composite?': indica con 'Yes' la configuración con el Composite Rank mínimo global.\n"
        " - 'Best per Model Composite?': indica con 'Yes' la mejor configuración dentro de cada modelo.\n\n"
        "Este análisis permite identificar la mejor configuración global y la mejor por modelo,\n"
        "teniendo en cuenta Accuracy, Time, Memory y Complexity.\n"
    )
    expl_sheet.write(0, 0, expl_text)
    expl_sheet.set_column(0, 0, 120)

    # =========================================================================
    # 6) HOJA "Charts": GRÁFICOS DINÁMICOS A PARTIR DE LOS DATOS CALCULADOS
    # =========================================================================
    charts_sheet = workbook.add_worksheet("Charts")
    charts_sheet.write(0, 0, "Visualización de Métricas por Configuración (ver 'Config Label' en Comparisons)")
    
    # Usamos la hoja Comparisons para generar gráficos
    total_rows = len(agrupado) + 1  # Incluye encabezado

    # --- Gráfico 1: Column Chart para Avg Accuracy ---
    chart1 = workbook.add_chart({'type': 'column'})
    chart1.add_series({
        'name': 'Avg Accuracy',
        'categories': f"=Comparisons!$I$2:$I${total_rows}",
        'values':     f"=Comparisons!$E$2:$E${total_rows}",
        'data_labels': {'value': True},
    })
    chart1.set_title({'name': 'Promedio Accuracy por Configuración'})
    chart1.set_x_axis({'name': 'Config Label'})
    chart1.set_y_axis({'name': 'Avg Accuracy'})
    chart1.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B3', chart1, {'x_scale': 1.2, 'y_scale': 1.2})
    
    # --- Gráfico 2: Column Chart para Avg Execution Time ---
    chart2 = workbook.add_chart({'type': 'column'})
    chart2.add_series({
        'name': 'Avg Execution Time (s)',
        'categories': f"=Comparisons!$I$2:$I${total_rows}",
        'values':     f"=Comparisons!$F$2:$F${total_rows}",
        'data_labels': {'value': True},
    })
    chart2.set_title({'name': 'Promedio Tiempo de Ejecución por Configuración'})
    chart2.set_x_axis({'name': 'Config Label'})
    chart2.set_y_axis({'name': 'Avg Time (s)'})
    chart2.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B20', chart2, {'x_scale': 1.2, 'y_scale': 1.2})
    
    # --- Gráfico 3: Column Chart para Avg Memory ---
    chart3 = workbook.add_chart({'type': 'column'})
    chart3.add_series({
        'name': 'Avg Memory (MB)',
        'categories': f"=Comparisons!$I$2:$I${total_rows}",
        'values':     f"=Comparisons!$G$2:$G${total_rows}",
        'data_labels': {'value': True},
    })
    chart3.set_title({'name': 'Promedio de Memoria por Configuración'})
    chart3.set_x_axis({'name': 'Config Label'})
    chart3.set_y_axis({'name': 'Avg Memory (MB)'})
    chart3.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B37', chart3, {'x_scale': 1.2, 'y_scale': 1.2})
    
    # --- Gráfico 4: Scatter Chart - Execution Time vs. Accuracy ---
    # Se crea una serie por cada configuración para mostrar la etiqueta completa
    scatter_time = workbook.add_chart({'type': 'scatter'})
    for row in range(2, total_rows+1):
        # Cada serie se obtiene de una fila de Comparisons
        serie_name = f"=Comparisons!$I${row}"
        x_value = f"=Comparisons!$F${row}"
        y_value = f"=Comparisons!$E${row}"
        scatter_time.add_series({
            'name': serie_name,
            'categories': x_value,
            'values': y_value,
            'marker': {'type': 'circle', 'size': 7},
            'data_labels': {'value': True},
        })
    scatter_time.set_title({'name': 'Tiempo vs. Accuracy (por Configuración)'})
    scatter_time.set_x_axis({'name': 'Avg Execution Time (s)'})
    scatter_time.set_y_axis({'name': 'Avg Accuracy'})
    scatter_time.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('K3', scatter_time, {'x_scale': 1.2, 'y_scale': 1.2})
    
    # --- Gráfico 5: Scatter Chart - Memory vs. Accuracy ---
    scatter_mem = workbook.add_chart({'type': 'scatter'})
    for row in range(2, total_rows+1):
        serie_name = f"=Comparisons!$I${row}"
        x_value = f"=Comparisons!$G${row}"  # Memory
        y_value = f"=Comparisons!$E${row}"  # Accuracy
        scatter_mem.add_series({
            'name': serie_name,
            'categories': x_value,
            'values': y_value,
            'marker': {'type': 'square', 'size': 7},
            'data_labels': {'value': True},
        })
    scatter_mem.set_title({'name': 'Memoria vs. Accuracy (por Configuración)'})
    scatter_mem.set_x_axis({'name': 'Avg Memory (MB)'})
    scatter_mem.set_y_axis({'name': 'Avg Accuracy'})
    scatter_mem.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('K20', scatter_mem, {'x_scale': 1.2, 'y_scale': 1.2})
    
    # =========================================================================
    # 7) CERRAR Y GUARDAR EL ARCHIVO EXCEL
    # =========================================================================
    workbook.close()
    print(f"Archivo Excel '{excel_filename}' creado exitosamente.")
    print("Hojas incluidas:")
    print(" - 'Raw Data': Datos crudos con valores derivados (Total Tests, Accuracy, Numeric Complexity).")
    print(" - 'Comparisons': Comparación de configuraciones con promedios, rankings y composite ranking.")
    print(" - 'Explanation': Detalle de variables y métodos de cálculo.")
    print(" - 'Charts': Gráficos nativos de Excel que muestran la información por configuración (model + configuración).")

if __name__ == "__main__":
    main()


Archivo Excel 'Analisis_JAIIO_FIJOS.xlsx' creado exitosamente.
Hojas incluidas:
 - 'Raw Data': Datos crudos con valores derivados (Total Tests, Accuracy, Numeric Complexity).
 - 'Comparisons': Comparación de configuraciones con promedios, rankings y composite ranking.
 - 'Explanation': Detalle de variables y métodos de cálculo.
 - 'Charts': Gráficos nativos de Excel que muestran la información por configuración (model + configuración).


: 